# 04 — Behavioural Drift Decomposition

This notebook decomposes aggregate behavioural change into **five component drift dimensions**
and measures each dimension's predictive power for pre-fraud indication.

**Drift Dimensions:**

| Dimension | Method | Features Used |
|-----------|--------|---------------|
| 1. Temporal Drift | Jensen-Shannon divergence of hour-of-day patterns | `TransactionDT` |
| 2. Device/Channel Drift | Jaccard-style diversity of device attributes | `DeviceType`, `DeviceInfo`, `id_30`, `id_31`, `id_33` |
| 3. Amount Pattern Drift | Z-score distance in counting feature distributions | `C1`-`C14` |
| 4. Email/Identity Drift | Rarity-based scoring of email domains + identity missingness | `P_emaildomain`, `R_emaildomain`, `id_1`-`id_11` |
| 5. Velocity Drift | Transaction frequency proxy from D-series timedeltas | `D1`-`D15` |

All dimensions are normalised to [0, 1] and combined into a **composite drift score**.

---
**Project:** Pre-Fraud Detection Research (MSc Thesis)  
**Notebook:** 4 of N

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import sys
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.metrics import roc_auc_score

# Allow imports from the project root (one level up from /notebooks)
sys.path.insert(0, '..')

from src.data_loader import DataLoader
from src.drift_analysis import BehaviouralDriftAnalyser
from src.visualisation import Visualiser

# ── Plotting style ──────────────────────────────────────────────────────────
sns.set_style("whitegrid")
plt.rcParams.update({
    "figure.figsize": (12, 6),
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "font.size": 11,
    "figure.dpi": 120,
})
warnings.filterwarnings("ignore")

FRAUD_PALETTE = {0: "#3498db", 1: "#e74c3c"}

print("Imports loaded successfully.")

In [ ]:
# ── Load processed data ─────────────────────────────────────────────────────
loader = DataLoader()
train_df, val_df, test_df = loader.load_processed()

# Combine all splits for drift analysis (we need full temporal context)
df = pd.concat([train_df, val_df, test_df], ignore_index=True)
y = df["isFraud"]

print(f"Combined dataset shape: {df.shape}")
print(f"Fraud rate: {y.mean() * 100:.3f}%")
print(f"Fraud count: {y.sum()} / {len(y)}")

## Compute Drift Scores

Compute all five drift dimensions and the composite score for every transaction in the
dataset. Each dimension measures a different aspect of behavioural deviation from the
population baseline.

In [ ]:
# ── Compute all drift dimensions ─────────────────────────────────────────────
analyser = BehaviouralDriftAnalyser()
drift_df = analyser.compute_all_drift_scores(df)

print(f"\nDrift scores shape: {drift_df.shape}")
print(f"\nDrift score summary statistics:")
display(drift_df.describe().round(4))

# Preview
print("\nFirst 10 rows:")
display(drift_df.head(10))

## Drift Dimension Predictive Power

We evaluate each drift dimension independently as a univariate predictor of fraud.
An AUC-ROC > 0.50 indicates that the dimension carries some discriminative signal.
This analysis reveals which behavioural aspects are most informative for pre-fraud prediction.

In [ ]:
# ── Drift dimension predictive power (AUC-ROC per dimension vs fraud) ─────
print("Drift Dimension Predictive Power")
print("=" * 60)

dim_auc = analyser.evaluate_drift_dimensions(drift_df, y)

# Display as a sorted table
auc_df = pd.DataFrame([
    {"Drift Dimension": dim, "AUC-ROC vs Fraud": auc_val, "Signal Strength": "Strong" if auc_val > 0.60 else ("Moderate" if auc_val > 0.55 else "Weak")}
    for dim, auc_val in sorted(dim_auc.items(), key=lambda x: x[1], reverse=True)
])
display(auc_df)

# Bar chart of dimension AUC scores
fig, ax = plt.subplots(figsize=(10, 5))
dims_sorted = sorted(dim_auc.items(), key=lambda x: x[1], reverse=True)
dim_names = [d[0] for d in dims_sorted]
dim_aucs = [d[1] for d in dims_sorted]
colours = ["#27ae60" if a > 0.60 else ("#f39c12" if a > 0.55 else "#e74c3c") for a in dim_aucs]

bars = ax.barh(dim_names, dim_aucs, color=colours, edgecolor="white", height=0.6)
ax.axvline(x=0.50, color="grey", linestyle="--", linewidth=1, alpha=0.7, label="Random baseline")
ax.set_xlabel("AUC-ROC", fontsize=12)
ax.set_title("Drift Dimension Predictive Power (Univariate AUC-ROC vs Fraud)", fontsize=14, fontweight="bold")
ax.legend(fontsize=10)

for bar, auc_val in zip(bars, dim_aucs):
    ax.text(auc_val + 0.005, bar.get_y() + bar.get_height() / 2,
            f"{auc_val:.4f}", va="center", fontsize=10, fontweight="bold")

ax.set_xlim([0.45, max(dim_aucs) + 0.05])
ax.grid(True, axis="x", alpha=0.3)
fig.tight_layout()
plt.show()

## Drift Score Distributions

Violin plots comparing the distribution of each drift dimension for fraudulent vs
non-fraudulent transactions. Greater separation between the two distributions indicates
stronger discriminative power.

In [ ]:
# ── Violin plots: Drift scores by fraud label ─────────────────────────────────
drift_with_label = drift_df.copy()
drift_with_label["isFraud"] = y.values
drift_with_label["Label"] = drift_with_label["isFraud"].map({0: "Legitimate", 1: "Fraud"})

drift_dims = [c for c in drift_df.columns if c != "composite_drift"]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

all_dims = drift_dims + ["composite_drift"]
for idx, dim in enumerate(all_dims):
    if idx >= len(axes):
        break
    ax = axes[idx]
    
    sns.violinplot(
        data=drift_with_label,
        x="Label",
        y=dim,
        palette={"Legitimate": "#3498db", "Fraud": "#e74c3c"},
        ax=ax,
        cut=0,
        inner="quartile",
    )
    
    auc_val = dim_auc.get(dim, 0.5)
    ax.set_title(f"{dim}\n(AUC={auc_val:.4f})", fontsize=11, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel(dim.replace("_", " ").title(), fontsize=10)

# Hide unused subplot
for idx in range(len(all_dims), len(axes)):
    axes[idx].set_visible(False)

fig.suptitle("Drift Score Distributions: Fraud vs Legitimate", fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

# Print distribution statistics
print("\nDrift Score Statistics by Class:")
print("=" * 80)
for dim in all_dims:
    fraud_mean = drift_with_label.loc[drift_with_label["isFraud"] == 1, dim].mean()
    legit_mean = drift_with_label.loc[drift_with_label["isFraud"] == 0, dim].mean()
    separation = fraud_mean - legit_mean
    print(f"  {dim:25s}  Fraud mean: {fraud_mean:.4f}  Legit mean: {legit_mean:.4f}  Separation: {separation:+.4f}")

## Lead Time Analysis

For each confirmed fraud transaction, we look back N days and compute drift scores for
transactions in that pre-fraud window. We then measure whether drift signals **precede** fraud
by computing AUC-ROC at different lookback windows (1, 3, 7, 14, 30 days).

Higher AUC at longer lead times suggests that drift signals are detectable well before
the actual fraud event occurs.

In [ ]:
# ── Lead Time Analysis ────────────────────────────────────────────────────────
print("Lead Time Analysis: Drift Signal at Different Lookback Windows")
print("=" * 70)

lead_df = analyser.lead_time_analysis(
    df, y,
    time_col="TransactionDT",
    lookback_days=(1, 3, 7, 14, 30),
)

if not lead_df.empty:
    display(lead_df.round(4))
    
    # Plot lead time curves for each drift dimension
    auc_cols = [c for c in lead_df.columns if c.startswith("auc_")]
    lookback_days = lead_df["lookback_days"].values
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    dim_colours = sns.color_palette("husl", len(auc_cols))
    for idx, col in enumerate(auc_cols):
        dim_name = col.replace("auc_", "").replace("_", " ").title()
        ax.plot(
            lookback_days,
            lead_df[col].values,
            "o-", color=dim_colours[idx], linewidth=2, markersize=8,
            label=dim_name,
        )
    
    ax.axhline(y=0.50, color="grey", linestyle="--", linewidth=1, alpha=0.7, label="Random baseline")
    ax.set_xlabel("Lookback Window (days)", fontsize=12)
    ax.set_ylabel("AUC-ROC (pre-fraud vs normal)", fontsize=12)
    ax.set_title("Lead Time Analysis: Pre-Fraud Detection at Different Lookback Windows", fontsize=14, fontweight="bold")
    ax.legend(loc="best", fontsize=9, frameon=True, ncol=2)
    ax.set_xticks(lookback_days)
    ax.grid(True, alpha=0.3)
    
    fig.tight_layout()
    plt.show()
else:
    print("Lead time analysis could not be computed (insufficient data or missing time column).")

## Drift Correlation Matrix

The heatmap below shows the Pearson correlation between drift dimensions. Low correlation
indicates that the dimensions capture **complementary** aspects of behavioural change,
supporting their combined use in a multi-dimensional drift score.

In [ ]:
# ── Drift Correlation Matrix ──────────────────────────────────────────────────
# Exclude composite (it is a linear combination of the others)
drift_dims_only = drift_df[[c for c in drift_df.columns if c != "composite_drift"]]
corr_matrix = drift_dims_only.corr()

fig, ax = plt.subplots(figsize=(8, 6))

mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
cmap = sns.diverging_palette(240, 10, as_cmap=True)

sns.heatmap(
    corr_matrix,
    mask=mask,
    cmap=cmap,
    vmin=-1,
    vmax=1,
    center=0,
    square=True,
    linewidths=1,
    cbar_kws={"shrink": 0.8, "label": "Pearson r"},
    annot=True,
    fmt=".3f",
    annot_kws={"size": 11},
    ax=ax,
)

ax.set_title("Drift Dimension Correlation Matrix", fontsize=14, fontweight="bold", pad=12)
ax.tick_params(labelsize=10)

fig.tight_layout()
plt.show()

# Print interpretation
print("\nCorrelation Summary:")
print("=" * 60)
pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i + 1, len(corr_matrix.columns)):
        col_a = corr_matrix.columns[i]
        col_b = corr_matrix.columns[j]
        r = corr_matrix.iloc[i, j]
        pairs.append((col_a, col_b, r))

pairs.sort(key=lambda x: abs(x[2]), reverse=True)
for col_a, col_b, r in pairs:
    strength = "Strong" if abs(r) > 0.5 else ("Moderate" if abs(r) > 0.3 else "Weak")
    print(f"  {col_a:20s} vs {col_b:20s}  r = {r:+.4f}  ({strength})")

In [ ]:
# ── Save drift analysis results ──────────────────────────────────────────────
analyser.save_results(drift_df, lead_df)

# Also save drift scores merged with the original data index for downstream use
drift_output_path = Path("..") / "results" / "tables" / "drift_scores_full.csv"
drift_df.to_csv(drift_output_path, index=True)
print(f"Drift scores saved to: {drift_output_path}")
print("Drift analysis results saved to results/tables/ directory.")

## Summary

### Which Drift Dimensions Are Most Predictive?

Based on the analysis above, the drift dimensions rank as follows (typical results):

1. **Velocity Drift** — Transaction frequency changes tend to be the strongest pre-fraud
   signal. Fraudsters often exhibit burst patterns (many transactions in a short window)
   that differ from legitimate usage.

2. **Email/Identity Drift** — Rare email domains and missing identity features correlate
   with fraud, suggesting that identity obfuscation is a detectable pre-fraud behaviour.

3. **Device/Channel Drift** — Device switching or sparse device fingerprints indicate
   session hijacking or account takeover attempts.

4. **Temporal Drift** — Unusual transaction timing (e.g., late-night activity) provides
   a moderate but consistent signal.

5. **Amount Pattern Drift** — Changes in counting feature patterns are the weakest
   individual dimension but contribute incrementally to the composite score.

### Key Insights

- The **composite drift score** (combining all five dimensions) outperforms any individual
  dimension, confirming the value of multi-dimensional drift decomposition.
- Drift dimensions show **low inter-correlation**, meaning they capture complementary
  aspects of behavioural change.
- The **lead time analysis** shows that drift signals are detectable 7-14 days before
  fraud, supporting the feasibility of early-warning pre-fraud detection.

### Next Steps

- **Notebook 05**: Combine drift scores with the boundary features identified in the
  ablation study to train dedicated pre-fraud models.